In [4]:
import pandas as pd
DATA_DIR= "../data"
import os
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import healpy as hp
import fitsio
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler

In [6]:
but = Butler("/global/cfs/cdirs/lsst/production/gen3/rubin/decam/repo/")

In [37]:
myDays = {"g":[],
"r":[],
"i":[],
"z":[],
         "all":[]}
for ref in but.query_all_datasets("DECam/raw/all",where="detector=3 AND instrument='DECam'"):
    myDays[ref.dataId['band']].append(ref.dataId['day_obs'])
    myDays["all"].append(ref.dataId['day_obs'])
for b in myDays.keys():
    myDays[b] = np.unique(myDays[b])

In [56]:
bias_days = np.loadtxt(os.path.join(DATA_DIR,"good_bias.txt"),dtype=int)
flat_days = np.loadtxt(os.path.join(DATA_DIR,"good_flat.txt"),dtype=int)

In [61]:
flatSpan = {"g":[],
"r":[],
"i":[],
"z":[]}

In [62]:
for b in flatSpan.keys():
    for band_day in myDays[b]:
        flatSpan[b].append(flat_days[band_day>flat_days][-1])
for b in flatSpan.keys():
    flatSpan[b] = np.unique(flatSpan[b])

In [73]:
for b in flatSpan.keys():
    
    with open(os.path.join(DATA_DIR,f"pilot_flat_{b}.txt"),"w+") as f:
        for x in flatSpan[b]:
            f.write(str(x))
            f.write("\n")

In [67]:
biasSpan = []
for band_day in myDays["all"]:
    biasSpan.append(bias_days[band_day>bias_days][-1])
biasSpan = np.unique(biasSpan)

In [71]:
with open(os.path.join(DATA_DIR,"pilot_bias.txt"),"w+") as f:
    for x in biasSpan:
        f.write(str(x))
        f.write("\n")

## Get the exposure numbers for the calibs

In [74]:
import requests
import os
from tqdm import tqdm

natroot = "https://astroarchive.noirlab.edu"
adsurl = f"{natroot}/api/adv_search"
DATA_DIR = "/global/homes/s/seanmacb/decam_template_tools/endurance/data"

In [75]:
x

np.int64(20181001)

#### Biases

In [102]:
def formatter(i):
    i=str(i)
    return f"{i[:4]}-{i[4:6]}-{i[6:]}"

In [133]:
for x in biasSpan:
    print(x)

20130902
20131001
20131101
20131201
20131231
20140101
20140113
20140801
20141001
20141101
20141201
20160802
20160901
20161005
20161006
20161031
20161120
20170115
20170702
20170831
20171006
20171101
20180815
20181001
20181101


In [135]:
response = requests.post(f'{adsurl}/find/?limit=1000', json=jj)
response = pd.DataFrame(response.json()[1:])

In [139]:
len(response)

0

In [141]:
for x in biasSpan:
    jj = {
        "outfields": [
            "md5sum",
            "archive_filename",
            "instrument",
            "proc_type",
            "obs_type",
        ],
        "search": [
            ["instrument", "decam"],
            ["proc_type", "raw"],
            # ["obs_type", "dome flat"],
            ["obs_type", "zero"],
            ["caldat", formatter(x),formatter(x)],  # The API natively understands this as a range
        ]
    }
    
    response = requests.post(f'{adsurl}/find/?limit=1000', json=jj)
    response = pd.DataFrame(response.json()[1:])
    if len(response)>0:
        print(f"{len(response)} bias exposures found for day {formatter(x)}.")
        with open(os.path.join(DATA_DIR,"pilot_bias_md5s.txt"),"a") as f:
            for x in response['md5sum']:
                f.write(x)
                f.write("\n")
    else:
        print(f"No bias exposures found for day {formatter(x)}. Continuing.")

36 bias exposures found for day 2013-09-02.
27 bias exposures found for day 2013-10-01.
89 bias exposures found for day 2013-11-01.
57 bias exposures found for day 2013-12-01.
27 bias exposures found for day 2013-12-31.
28 bias exposures found for day 2014-01-01.
27 bias exposures found for day 2014-01-13.
22 bias exposures found for day 2014-08-01.
33 bias exposures found for day 2014-10-01.
87 bias exposures found for day 2014-11-01.
No bias exposures found for day 2014-12-01. Continuing.
22 bias exposures found for day 2016-08-02.
36 bias exposures found for day 2016-09-01.
33 bias exposures found for day 2016-10-05.
32 bias exposures found for day 2016-10-06.
33 bias exposures found for day 2016-10-31.
44 bias exposures found for day 2016-11-20.
22 bias exposures found for day 2017-01-15.
33 bias exposures found for day 2017-07-02.
33 bias exposures found for day 2017-08-31.
24 bias exposures found for day 2017-10-06.
22 bias exposures found for day 2017-11-01.
22 bias exposures fo

#### Flats

In [154]:
filters = {"y":'Y DECam c0005 10095.0 1130.0', 
           "g":'g DECam SDSS c0001 4720.0 1520.0',
           "i":'i DECam SDSS c0003 7835.0 1470.0',
           "r":'r DECam SDSS c0002 6415.0 1480.0', 
           "u":'u DECam c0006 3500.0 1000.0',
           "z":'z DECam SDSS c0004 9260.0 1520.0'
          }

In [160]:
for b,dates in flatSpan.items():
    print(f"Band {b}.")
    print(10*"=====")
    for d in dates:
        jj = {
            "outfields": [
                "md5sum",
                "archive_filename",
                "instrument",
                "proc_type",
                "obs_type",
                "FILTER"
            ],
            "search": [
                ["instrument", "decam"],
                ["proc_type", "raw"],
                ["obs_type", "dome flat"],
                ["FILTER", filters[b]],
                ["caldat", formatter(d),formatter(d)],  # The API natively understands this as a range
            ]
        }
        
        response = requests.post(f'{adsurl}/find/?limit=1000', json=jj)
        response = pd.DataFrame(response.json()[1:])
        if len(response)>0:
            print(f"{len(response)} flat exposures found for day {formatter(x)}.")
            with open(os.path.join(DATA_DIR,f"{b}_pilot_flat_md5s.txt"),"a") as f:
                for x in response['md5sum']:
                    f.write(x)
                    f.write("\n")
        else:
            print(f"No flat exposures found for day {formatter(x)}. Continuing.")

Band g.
10 flat exposures found for day 52f9-da-e2bc3f4db99d661c7612542a15.
No flat exposures found for day 3879-9e-63fdb45ceb8503c9a4f74c51b8. Continuing.
11 flat exposures found for day 3879-9e-63fdb45ceb8503c9a4f74c51b8.
10 flat exposures found for day 13cd-55-e3a386efdd87c6ef9bbb527233.
11 flat exposures found for day b687-08-42f65a1caaa7af4d1cd5a21c0b.
11 flat exposures found for day ff61-45-828fa09cf6a20e3192c22648ef.
11 flat exposures found for day 0ed6-04-52c1243274ce29d40d282ccf7f.
No flat exposures found for day 63b2-c8-2c2c34aef85fa1eced2e52b4aa. Continuing.
11 flat exposures found for day 63b2-c8-2c2c34aef85fa1eced2e52b4aa.
11 flat exposures found for day 88ca-56-f3d520eb838ad73f6abe0d1ece.
11 flat exposures found for day 1619-f6-17a48aaec2dd5b2070bc994d66.
11 flat exposures found for day 6d48-cd-75cff02cd5322de385c9be3e20.
22 flat exposures found for day 93a8-af-a3b39f7fb19dc94a4449462596.
11 flat exposures found for day 5a40-ac-cab27d28d12ef329b5c9f1836c.
11 flat exposure